## 1. Installation and Imports




In [1]:
import imaplib
import email
import re
import pickle
import os
from datetime import datetime, timedelta
from email.header import decode_header
from email.utils import parsedate_to_datetime
import schedule
import time
import threading

# ML imports
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

print("All imports successful!")


All imports successful!


## 2. Configuration



In [2]:
# IMAP Configuration
IMAP_SERVER = "imap.gmail.com"
IMAP_PORT = 993

# Load Gmail credentials from environment variables
EMAIL_ADDRESS = os.getenv("GMAIL_EMAIL_ADDRESS", "")
EMAIL_PASSWORD = os.getenv("GMAIL_APP_PASSWORD", "")

# Model Configuration
MODEL_FILE = 'spam_classifier_model.pkl'
VECTORIZER_FILE = 'spam_vectorizer.pkl'

# Email Check Configuration
CHECK_INTERVAL_HOURS = 6
MAX_EMAILS_TO_PROCESS = 50  # Process last 50 emails

# Training Data Configuration (if training model from scratch)
TRAIN_MODEL = False  # Set to True to retrain the model


## 3. IMAP Connection




In [3]:
def connect_imap():
    """Connect to Gmail via IMAP and return connection"""
    try:
        # Connect to IMAP server
        mail = imaplib.IMAP4_SSL(IMAP_SERVER, IMAP_PORT)
        
        # Login
        mail.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        
        # Select inbox
        mail.select('INBOX')
        
        print(f"✅ Connected to {EMAIL_ADDRESS}")
        return mail
    except imaplib.IMAP4.error as e:
        print(f"❌ IMAP connection failed: {e}")
        print("Make sure you've:")
        print("1. Enabled 2-step verification on your Google account")
        print("2. Generated an App Password at: https://myaccount.google.com/apppasswords")
        print("3. Entered the correct email and app password in the configuration cell")
        return None

print("IMAP connection function ready!")


IMAP connection function ready!


## 4. Email Preprocessing




In [4]:
def preprocess_email(text):
    """Clean and preprocess email text"""
    if not text:
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove special characters except spaces
    text = re.sub(r'[^\w\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def extract_email_content(msg_email):
    """Extract text content from email message (IMAP format)"""
    try:
        body = ""
        subject = ""
        
        # Get subject
        if msg_email['Subject']:
            subject_header = decode_header(msg_email['Subject'])
            subject = "".join([
                text.decode(encoding or 'utf-8', errors='ignore') 
                if isinstance(text, bytes) else text
                for text, encoding in subject_header
            ])
        
        # Get body
        if msg_email.is_multipart():
            for part in msg_email.walk():
                content_type = part.get_content_type()
                content_disposition = str(part.get("Content-Disposition"))
                
                # Skip attachments
                if "attachment" not in content_disposition:
                    if content_type == "text/plain":
                        try:
                            part_body = part.get_payload(decode=True)
                            if part_body:
                                body = part_body.decode('utf-8', errors='ignore')
                                break
                        except:
                            pass
                    elif content_type == "text/html":
                        try:
                            html_body = part.get_payload(decode=True)
                            if html_body:
                                # Simple HTML tag removal
                                body = re.sub('<[^<]+?>', '', html_body.decode('utf-8', errors='ignore'))
                        except:
                            pass
        else:
            # Single part message
            try:
                body = msg_email.get_payload(decode=True)
                if body:
                    body = body.decode('utf-8', errors='ignore')
                    # Remove HTML tags if present
                    if '<html' in body.lower() or '<body' in body.lower():
                        body = re.sub('<[^<]+?>', '', body)
            except:
                pass
        
        full_text = f"{subject} {body}"
        return preprocess_email(full_text)
    except Exception as e:
        print(f"Error extracting email content: {e}")
        return ""

print("Email preprocessing functions ready!")


Email preprocessing functions ready!


## 5. Spam Classification Model



In [5]:
def train_spam_model():
    """Train a spam classification model with sample data"""
    # Sample training data (in production, use larger dataset)
    spam_samples = [
        "Congratulations! You've won $1,000,000! Click here to claim your prize!",
        "Get rich quick! Make money from home with this amazing opportunity!",
        "URGENT: Your account will be suspended. Verify your details now!",
        "Free Viagra! Buy now at amazing prices. Limited time offer!",
        "You are the winner! Collect your prize money today!",
        "Nigerian Prince needs your help. Send money to claim millions!",
        "Click this link to see shocking photos!",
        "Your email has been selected for a free gift card!",
        "Lottery winner! Claim your $500,000 prize now!",
        "Make $1000 a day working from home!",
        "Hot singles in your area want to meet you!",
        "Account verification required. Click to avoid suspension!",
    ]
    
    ham_samples = [
        "Thank you for your purchase. Your order will be delivered soon.",
        "Meeting scheduled for tomorrow at 2 PM. See you there!",
        "Attached is the report you requested. Please review when convenient.",
        "Happy birthday! Hope you have a wonderful day!",
        "Reminder: Team meeting at 3 PM today in conference room.",
        "Your reservation has been confirmed for Friday evening.",
        "Dinner plans? Let me know what time works for you.",
        "Great work on the presentation! The client was very impressed.",
        "Can you review the proposal I sent earlier this week?",
        "Weekend trip to the beach sounds perfect!",
        "Your subscription will renew on the 15th of next month.",
        "Project timeline updated. Please check the shared document.",
    ]
    
    # Prepare data
    X = spam_samples + ham_samples
    y = [1] * len(spam_samples) + [0] * len(ham_samples)  # 1 = spam, 0 = ham
    
    # Preprocess
    X = [preprocess_email(text) for text in X]
    
    # Create vectorizer
    vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
    X_vectorized = vectorizer.fit_transform(X)
    
    # Train model
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_vectorized, y)
    
    # Save model and vectorizer
    with open(MODEL_FILE, 'wb') as f:
        pickle.dump(model, f)
    
    with open(VECTORIZER_FILE, 'wb') as f:
        pickle.dump(vectorizer, f)
    
    # Evaluate
    y_pred = model.predict(X_vectorized)
    accuracy = accuracy_score(y, y_pred)
    
    print(f"Model trained! Training accuracy: {accuracy:.2f}")
    print("\nClassification Report:")
    print(classification_report(y, y_pred, target_names=['Ham', 'Spam']))
    
    return model, vectorizer

def load_spam_model():
    """Load trained spam classification model"""
    if not os.path.exists(MODEL_FILE) or not os.path.exists(VECTORIZER_FILE):
        print("Model not found. Training new model...")
        model, vectorizer = train_spam_model()
    else:
        with open(MODEL_FILE, 'rb') as f:
            model = pickle.load(f)
        with open(VECTORIZER_FILE, 'rb') as f:
            vectorizer = pickle.load(f)
        print("Model loaded successfully!")
    
    return model, vectorizer

# Load or train model
classifier_model, text_vectorizer = load_spam_model()


Model not found. Training new model...
Model trained! Training accuracy: 1.00

Classification Report:
              precision    recall  f1-score   support

         Ham       1.00      1.00      1.00        12
        Spam       1.00      1.00      1.00        12

    accuracy                           1.00        24
   macro avg       1.00      1.00      1.00        24
weighted avg       1.00      1.00      1.00        24



In [6]:
def classify_email(email_text, model=None, vectorizer=None):
    """Classify an email as spam or ham"""
    if model is None:
        model = classifier_model
    if vectorizer is None:
        vectorizer = text_vectorizer
    
    try:
        processed_text = preprocess_email(email_text)
        if not processed_text:
            return 0, 0.5  # Default to ham if empty
        
        vectorized = vectorizer.transform([processed_text])
        prediction = model.predict(vectorized)[0]
        probability = model.predict_proba(vectorized)[0]
        
        return prediction, max(probability)
    except Exception as e:
        print(f"Error classifying email: {e}")
        return 0, 0.5  # Default to ham on error

print("Classification function ready!")


Classification function ready!


## 6. Fetch and Process Emails



In [7]:
def get_email_list(mail, max_results=MAX_EMAILS_TO_PROCESS):
    """Fetch list of emails from inbox"""
    try:
        # Search for all emails in inbox
        status, messages = mail.search(None, 'ALL')
        
        if status != 'OK':
            print("Error searching emails")
            return []
        
        email_ids = messages[0].split()[-max_results:]  # Get last N emails
        return email_ids
    except Exception as e:
        print(f"Error fetching email list: {e}")
        return []

def get_email_content(mail, email_id):
    """Get full email content by message ID (IMAP)"""
    try:
        # Fetch email
        status, msg_data = mail.fetch(email_id, '(RFC822)')
        
        if status != 'OK':
            return None
        
        # Parse email
        email_body = msg_data[0][1]
        msg_email = email.message_from_bytes(email_body)
        
        return msg_email
    except Exception as e:
        print(f"Error fetching email content: {e}")
        return None

def move_email_to_spam(mail, email_id):
    """Move email to spam folder"""
    try:
        # List mailboxes to find spam folder
        status, folders = mail.list()
        
        spam_folder = None
        # Try common spam folder names
        for folder in folders:
            folder_str = folder.decode('utf-8')
            if 'spam' in folder_str.lower():
                # Extract folder name
                spam_folder = folder_str.split(' "/" ')[-1]
                break
        
        if spam_folder:
            # Copy email to spam folder
            mail.copy(email_id, spam_folder)
        else:
            # Try default Gmail spam folder
            try:
                mail.copy(email_id, '[Gmail]/Spam')
            except:
                mail.copy(email_id, 'SPAM')
        
        # Mark original as deleted
        mail.store(email_id, '+FLAGS', '\\Deleted')
        
        # Expunge to permanently delete from inbox
        mail.expunge()
        
        return True
    except Exception as e:
        print(f"Error moving email to spam: {e}")
        return False

print("Email processing functions ready!")


Email processing functions ready!


## 7. Main Email Check Function


In [8]:
def check_and_move_spam_emails():
    """Main function to check emails and move spam to spam folder"""
    print(f"\n[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Starting email check...")
    
    mail = None
    try:
        # Connect to IMAP
        mail = connect_imap()
        if not mail:
            print("❌ Failed to connect to Gmail")
            return
        
        # Get list of emails
        email_ids = get_email_list(mail)
        print(f"Found {len(email_ids)} emails to check")
        
        spam_count = 0
        processed_count = 0
        
        for email_id in email_ids:
            try:
                # Get email content
                msg_email = get_email_content(mail, email_id)
                if not msg_email:
                    continue
                
                # Extract and preprocess content
                email_text = extract_email_content(msg_email)
                
                # Classify email
                is_spam, confidence = classify_email(email_text)
                
                # Get subject for display
                subject = ""
                if msg_email['Subject']:
                    subject_header = decode_header(msg_email['Subject'])
                    subject = "".join([
                        text.decode(encoding or 'utf-8', errors='ignore') 
                        if isinstance(text, bytes) else text
                        for text, encoding in subject_header
                    ])
                
                if is_spam:
                    print(f"\n🚨 SPAM DETECTED: {subject[:50]}... (Confidence: {confidence:.2f})")
                    
                    # Move to spam
                    if move_email_to_spam(mail, email_id):
                        print(f"✅ Moved to spam folder")
                        spam_count += 1
                    else:
                        print(f"❌ Failed to move to spam folder")
                else:
                    print(f".", end="", flush=True)  # Progress indicator for ham emails
                
                processed_count += 1
                
            except Exception as e:
                print(f"Error processing email: {e}")
                continue
        
        print(f"\n\n✅ Check complete! Processed {processed_count} emails, moved {spam_count} to spam.")
        
    except Exception as e:
        print(f"❌ Error during email check: {e}")
    finally:
        # Close connection
        if mail:
            try:
                mail.close()
                mail.logout()
            except:
                pass

print("Main check function ready!")


Main check function ready!


## 9. Automated Scheduler Setup



In [9]:
def run_scheduler():
    """Run the email check scheduler"""
    # Schedule email check every 6 hours
    schedule.every(CHECK_INTERVAL_HOURS).hours.do(check_and_move_spam_emails)
    
    print(f"Scheduler started! Checking emails every {CHECK_INTERVAL_HOURS} hours.")
    print("Press Ctrl+C to stop the scheduler.")
    
    # Run initial check
    print("\nRunning initial check...")
    check_and_move_spam_emails()
    
    # Keep scheduler running
    try:
        while True:
            schedule.run_pending()
            time.sleep(60)  # Check every minute if any job is due
    except KeyboardInterrupt:
        print("\n\nScheduler stopped.")

print("Scheduler function ready!")


Scheduler function ready!


In [10]:
# Start the automated scheduler
# Uncomment to run the scheduler
# This will run continuously, checking emails every 6 hours

# run_scheduler()


## 10. Test Email Classification (Optional)


In [11]:
# Test the classification model with sample emails

test_emails = [
    "Congratulations! You won $1,000,000. Click here now!",
    "Thank you for your purchase. Your order has been shipped.",
    "Free money! No strings attached. Claim your prize today!",
    "Meeting tomorrow at 2 PM. Don't forget!",
    "URGENT: Your account will be suspended. Verify immediately!",
]

print("Testing email classification:")
print("=" * 60)

for email_text in test_emails:
    is_spam, confidence = classify_email(email_text)
    result = "SPAM" if is_spam else "HAM"
    emoji = "🚨" if is_spam else "✅"
    print(f"{emoji} {result:4s} ({confidence:.2f}): {email_text}")


Testing email classification:
🚨 SPAM (0.59): Congratulations! You won $1,000,000. Click here now!
✅ HAM  (0.59): Thank you for your purchase. Your order has been shipped.
🚨 SPAM (0.61): Free money! No strings attached. Claim your prize today!
✅ HAM  (0.63): Meeting tomorrow at 2 PM. Don't forget!
🚨 SPAM (0.59): URGENT: Your account will be suspended. Verify immediately!
